# 6. Transfer audit for the stations I would act on

I did not audit every station. I limited this to the priority set and the Kitchener case stations, and to the direction that matters for a morning trip into Toronto: bus arrival, then train toward Union. Waits are capped at 60 minutes in the SQL, so a station with no midday train does not produce an unbounded gap. The cap is a reporting choice, and a median that sits on 60 minutes means many connections did not find a train inside the hour.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)
from scorecard.pipeline import load
table, recs, meta = load()

import pandas as pd
from scorecard.config import AM_PEAK, MIDDAY, PM_PEAK, EVENING, KITCHENER_CASE
waits = pd.read_parquet(ROOT / "data" / "processed" / "transfer_waits.parquet")
ids = set(table.loc[table.quadrant.eq("priority"), "station_id"]) | set(KITCHENER_CASE)
am = waits[(waits.direction.eq("bus_to_train")) & (waits.station_id.isin(ids)) & (waits.event_sec.between(*AM_PEAK))]
summary = am.groupby("station_id").wait_sec.agg(n="size", median_min=lambda s: s.median()/60, p90_min=lambda s: s.quantile(0.9)/60)
summary = summary.join(table.set_index("station_id")[["station_name","main_feeder_agency","primary_likely"]])
print(summary.sort_values("median_min", ascending=False).to_string())

              n  median_min    p90_min                  station_name main_feeder_agency primary_likely
station_id                                                                                            
RI           91   27.900000  60.000000                 Richmond Hill                yrt        service
OL           18   27.708333  60.000000                    Old Cummer                ttc        service
OR           16   26.891667  60.000000                        Oriole                ttc       land_use
LA           48   26.158333  60.000000                     Langstaff                yrt        service
MR           44   22.200000  32.328333                       Markham                ttc        parking
KI           85   22.000000  60.000000                     Kitchener            grt_bus           fare
GL          237   19.000000  29.180000                Guelph Central             guelph           fare
MK          227   18.666667  46.223333                      Milliken     